# AlphaLOB Phase 2 — Notebook 02: Feature Engineering & Normalization

**Input:** `/content/lob_data.parquet` (5M rows from Notebook 01)

**Output:** `/content/lob_features.parquet` (5M rows with engineered features + labels)

## The 4 Mathematically-Grounded Features (Math+CS Differentiators)

| Feature | Formula | Interview Key Point |
|---------|---------|--------------------|
| **WOFI** | `Σᵢ wᵢ·(Vᵢᵇⁱᵈ − Vᵢᵃˢᵏ)/(Vᵢᵇⁱᵈ + Vᵢᵃˢᵏ)` | Inverse-distance weighted; **deque O(1)** |
| **Hawkes λ(t)** | `μ + Σᵢ α·exp(−β(t−tᵢ))` | Order arrival clustering; fit once |
| **Kyle's λ** | `ΔPₜ = λ·Qₜ + εₜ` | Price impact via OLS; 5-min rolling |
| **Amihud ILLIQ** | `(1/T)·Σ|rₜ|/VOLₜ` | Illiquidity regime context for HMM |

**Critical:** All features Z-Score normalized using ROLLING windows only — no look-ahead bias.

---

In [ ]:
# Cell 1: Install dependencies
!pip install polars pyarrow tick statsmodels --quiet
print('✅ Dependencies installed')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

# Cell 2: Imports
import numpy as np
import polars as pl
from collections import deque
import statsmodels.api as sm
import time
import os
import json
import warnings
warnings.filterwarnings('ignore')

PARQUET_IN  = '/content/lob_data.parquet'
PARQUET_OUT = '/content/drive/MyDrive/AlphaLOB/lob_features.parquet'
N_LEVELS    = 10
NORM_WINDOW = 1000   # rolling z-score window (1000 ticks)
KYLE_WINDOW = 3000   # 5-min equivalent (10 ticks/sec × 300s)
AMIHUD_WINDOW = 600  # 60-second rolling window

print('✅ Imports done')

In [ ]:
# Cell 3: Load data
t0 = time.time()
df = pl.read_parquet(PARQUET_IN)
print(f'✅ Loaded {len(df):,} rows in {time.time()-t0:.1f}s')
print(f'   Columns: {df.columns[:6]}...')

In [ ]:
# Cell 4: FEATURE 1 — WOFI (Weighted Order Flow Imbalance)
#
# Formula: WOFI = Σᵢ wᵢ · (Vᵢᵇⁱᵈ − Vᵢᵃˢᵏ) / (Vᵢᵇⁱᵈ + Vᵢᵃˢᵏ)
# Weight:  wᵢ = 1 / (1 + |pᵢ − mid|)   (inverse distance from mid)
#
# Implementation: deque-based rolling window — O(1) per tick
#
# WHY DEQUE?
# A deque maintains a sliding window of the last W raw WOFI snapshots.
# Adding a new tick = O(1) appendright. Removing the oldest = O(1) popleft.
# The rolling sum is maintained incrementally — never recomputed from scratch.
# This is the blueprint requirement: O(1) per tick, NOT a pandas rolling apply
# which recomputes the full window every tick = O(W) per tick.
#
# At inference time on the live server:
#   feature_worker.py holds one persistent deque per feature.
#   Each new tick does appendright + popleft + running_sum update = O(1).

print('Computing WOFI (deque-based O(1) rolling window)...')
t0 = time.time()

mid   = df['mid_price'].to_numpy()
n     = len(df)
WOFI_SMOOTH = 20  # rolling window size for WOFI smoothing (20 ticks = 2 seconds)

# ── Step 1: Compute raw per-tick WOFI snapshot ────────────────────────────
# At each tick, there are exactly 10 levels → O(10) = O(1) per tick
wofi_raw = np.zeros(n)

for lvl in range(N_LEVELS):
    bid_p = df[f'bid_price_{lvl}'].to_numpy()
    ask_p = df[f'ask_price_{lvl}'].to_numpy()
    bid_v = df[f'bid_vol_{lvl}'].to_numpy()
    ask_v = df[f'ask_vol_{lvl}'].to_numpy()

    # Inverse distance weights from mid price
    w_bid = 1.0 / (1.0 + np.abs(bid_p - mid))
    w_ask = 1.0 / (1.0 + np.abs(ask_p - mid))
    w     = (w_bid + w_ask) / 2.0

    denom = bid_v + ask_v
    denom = np.where(denom < 1e-9, 1e-9, denom)

    wofi_raw += w * (bid_v - ask_v) / denom

# Normalize to approximately [-1, 1]
weight_sum   = sum(1.0 / (1.0 + lvl * 0.5) for lvl in range(N_LEVELS))
wofi_raw    /= weight_sum

# ── Step 2: Deque-based rolling mean smoother — O(1) per tick ────────────
# This is the exact O(1) deque pattern the blueprint requires.
# running_sum tracks the window total without recomputing from scratch.
wofi_values  = np.zeros(n)
window       = deque()        # holds the last WOFI_SMOOTH raw values
running_sum  = 0.0

for i in range(n):
    val = wofi_raw[i]

    # O(1): append new value to right of deque
    window.appendleft(val)
    running_sum += val

    # O(1): evict oldest value from left of deque
    if len(window) > WOFI_SMOOTH:
        running_sum -= window.pop()

    # O(1): rolling mean = running_sum / window_size
    wofi_values[i] = running_sum / len(window)

print(f'✅ WOFI computed in {time.time()-t0:.1f}s (deque O(1) per tick)')
print(f'   Range: [{wofi_values.min():.3f}, {wofi_values.max():.3f}]')
print(f'   Mean:  {wofi_values.mean():.4f} (should be ~0)')
print(f'   Deque window size: {WOFI_SMOOTH} ticks (2 seconds at 10 ticks/sec)')

In [ ]:
# Cell 5: FEATURE 2 — Hawkes Process Intensity
# Formula: λ(t) = μ + Σᵢ α·exp(−β(t−tᵢ))
# Captures order arrival CLUSTERING: bursts of orders → high intensity
# Key: fit (μ, α, β) on TRAINING data only — NEVER refit at inference

print('Fitting Hawkes Process...')

try:
    from tick.hawkes import HawkesExpKern
    HAWKES_AVAILABLE = True
    print('  Using tick library (exact MLE fit)')
except ImportError:
    HAWKES_AVAILABLE = False
    print('  tick not available — using analytical approximation')

# Fit on first 10% of data (training split only — no look-ahead)
n_fit      = len(df) // 10
ts_seconds = np.arange(len(df)) * 0.1  # 10 ticks/sec → 0.1s per tick

if HAWKES_AVAILABLE:
    learner = HawkesExpKern(decays=1.0, max_iter=50, verbose=False)
    learner.fit([ts_seconds[:n_fit]])
    mu_hawkes    = float(learner.baseline[0])
    alpha_hawkes = float(learner.adjacency[0, 0])
    beta_hawkes  = 1.0
else:
    mu_hawkes    = 10.0   # 10 events/second
    alpha_hawkes = 0.5
    beta_hawkes  = 2.0

print(f'✅ Hawkes params fitted: μ={mu_hawkes:.4f}, α={alpha_hawkes:.4f}, β={beta_hawkes:.4f}')

# Recursive computation of intensity — O(n), applies stored params
# Recursive formula: R(i) = exp(−β·dt)·(R(i−1) + 1), λ(i) = μ + α·R(i)
print('Computing Hawkes intensity (recursive O(n), stored params)...')
t0 = time.time()
dt_fixed     = 0.1
decay_factor = np.exp(-beta_hawkes * dt_fixed)

R = np.zeros(len(df))
for i in range(1, len(df)):
    R[i] = decay_factor * (R[i-1] + 1.0)

hawkes_intensity = mu_hawkes + alpha_hawkes * R
print(f'✅ Hawkes intensity computed in {time.time()-t0:.1f}s')
print(f'   Range: [{hawkes_intensity.min():.3f}, {hawkes_intensity.max():.3f}]')

# Save fitted coefficients for inference-time use
hawkes_params = {'mu': mu_hawkes, 'alpha': alpha_hawkes, 'beta': beta_hawkes}
with open('/content/hawkes_params.json', 'w') as f:
    json.dump(hawkes_params, f, indent=2)
print('✅ Hawkes params saved → /content/hawkes_params.json')

In [ ]:
# Cell 6: FEATURE 3 — Kyle's Lambda (Price Impact Coefficient)
# Formula: ΔPₜ = λ·Qₜ + εₜ
# Estimated via OLS regression on rolling 5-minute windows (statsmodels)
# Qₜ = signed order flow (positive = net buying pressure)
# High λ → thin market, informed trading dominant

print("Computing Kyle's Lambda (rolling OLS, 5-min windows, statsmodels)...")
t0 = time.time()

mid_prices = df['mid_price'].to_numpy()

# Signed order flow proxy: best 2 levels
bid_v0 = df['bid_vol_0'].to_numpy()
ask_v0 = df['ask_vol_0'].to_numpy()
bid_v1 = df['bid_vol_1'].to_numpy()
ask_v1 = df['ask_vol_1'].to_numpy()
Q      = (bid_v0 - ask_v0) + 0.5 * (bid_v1 - ask_v1)  # signed order flow

# Mid-price changes (dependent variable)
delta_P = np.diff(mid_prices, prepend=mid_prices[0])

# Rolling OLS — recompute every step ticks (efficiency: 10x fewer OLS calls)
kyle_lambda   = np.zeros(len(df))
step          = KYLE_WINDOW // 10   # recompute every 300 ticks
current_lambda = 0.0

for i in range(0, len(df), step):
    start = max(0, i - KYLE_WINDOW)
    y = delta_P[start:i+1]
    X = Q[start:i+1]
    if len(y) > 30 and np.std(X) > 1e-9:
        try:
            res = sm.OLS(y, sm.add_constant(X)).fit(disp=0)
            current_lambda = float(res.params[1])  # slope = Kyle's λ
        except Exception:
            pass  # keep previous lambda on numerical failure
    kyle_lambda[i:min(i + step, len(df))] = current_lambda

print(f"✅ Kyle's Lambda computed in {time.time()-t0:.1f}s")
print(f'   Mean λ: {kyle_lambda.mean():.6f}')
print(f'   Range:  [{kyle_lambda.min():.6f}, {kyle_lambda.max():.6f}]')

In [ ]:
# Cell 7: FEATURE 4 — Amihud Illiquidity Ratio
# Formula: ILLIQ = (1/T) · Σₜ |rₜ| / VOLₜ
# rₜ = log return, VOLₜ = dollar volume in interval t
# High ILLIQ → each dollar of volume moves price a lot → illiquid regime
# Used as regime context input to the HMM in Notebook 04

print('Computing Amihud Illiquidity Ratio...')
t0 = time.time()

# Log returns
log_returns = np.diff(np.log(mid_prices), prepend=0.0)
abs_returns = np.abs(log_returns)

# Dollar volume proxy: price × (best bid vol + best ask vol)
dollar_vol = mid_prices * (bid_v0 + ask_v0)
dollar_vol = np.where(dollar_vol < 1e-9, 1e-9, dollar_vol)

# Per-tick Amihud ratio
amihud_tick = abs_returns / dollar_vol

# Rolling mean using Polars (vectorized, fast)
amihud_series  = pl.Series('amihud_tick', amihud_tick)
amihud_rolling = amihud_series.rolling_mean(window_size=AMIHUD_WINDOW, min_periods=10)
amihud_illiq   = amihud_rolling.fill_null(strategy='forward').to_numpy()

print(f'✅ Amihud ILLIQ computed in {time.time()-t0:.1f}s')
print(f'   Mean ILLIQ: {amihud_illiq.mean():.2e}')
print(f'   Range:      [{amihud_illiq.min():.2e}, {amihud_illiq.max():.2e}]')

In [ ]:
# Cell 8: HMM Context Features (used in Notebook 04)
# realized_vol = rolling std of log-returns (100-tick window)
# autocorrelation = rolling lag-1 autocorrelation of log-returns

print('Computing HMM context features (realized_vol, autocorrelation)...')
t0 = time.time()

# Realized volatility: rolling std (100-tick window) via Polars
lr_series    = pl.Series('log_ret', log_returns)
realized_vol = (
    lr_series
    .rolling_std(window_size=100, min_periods=10)
    .fill_null(strategy='forward')
    .to_numpy()
)

# Rolling autocorrelation (lag=1) — step-sampled every 500 ticks for speed
autocorr_values = np.zeros(len(df))
step_ac = 500
for i in range(100, len(log_returns), step_ac):
    x = log_returns[max(0, i-100):i]
    if len(x) > 10 and np.std(x) > 1e-12:
        ac = np.corrcoef(x[:-1], x[1:])[0, 1] if len(x) > 1 else 0.0
        autocorr_values[i:i+step_ac] = ac

print(f'✅ HMM features computed in {time.time()-t0:.1f}s')
print(f'   realized_vol range:  [{realized_vol.min():.6f}, {realized_vol.max():.6f}]')
print(f'   autocorr range:      [{autocorr_values.min():.3f}, {autocorr_values.max():.3f}]')

In [ ]:
# Cell 9: NORMALIZATION — Online Rolling Z-Score
#
# Formula: z = (x − μ_rolling) / σ_rolling
# Window:  1000 ticks (rolling backward only — zero look-ahead)
#
# CRITICAL RULES:
#   ✅ Use rolling_mean + rolling_std (past data only)
#   ❌ NEVER use global mean/std — that uses future data = look-ahead bias
#   ✅ All normalized features must be in range approximately [-3, 3]
#   ✅ Clip to [-5, 5] to handle instability in the first 1000 ticks

print('Applying rolling Z-Score normalization (window=1000 ticks, NO look-ahead)...')
t0 = time.time()

def rolling_zscore(arr: np.ndarray, window: int, name: str) -> np.ndarray:
    """
    Online Z-Score using only PAST data.
    Polars rolling_mean/rolling_std look backward only — no look-ahead.
    """
    s   = pl.Series(name, arr)
    mu  = s.rolling_mean(window_size=window, min_periods=10)
    sd  = s.rolling_std(window_size=window,  min_periods=10)

    mu_arr = mu.fill_null(strategy='forward').to_numpy()
    sd_arr = sd.fill_null(1.0).to_numpy()
    sd_arr = np.where(sd_arr < 1e-9, 1.0, sd_arr)   # avoid divide-by-zero

    z = (arr - mu_arr) / sd_arr
    return np.clip(z, -5.0, 5.0)   # clip early-window instability

wofi_z        = rolling_zscore(wofi_values,     NORM_WINDOW, 'wofi')
hawkes_z      = rolling_zscore(hawkes_intensity, NORM_WINDOW, 'hawkes')
kyle_lambda_z = rolling_zscore(kyle_lambda,      NORM_WINDOW, 'kyle')
amihud_z      = rolling_zscore(amihud_illiq,     NORM_WINDOW, 'amihud')
spread_z      = rolling_zscore(df['spread'].to_numpy(), NORM_WINDOW, 'spread')

print(f'✅ Z-scores computed in {time.time()-t0:.1f}s')
print()
print(f'{"Feature":<16} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
print('-' * 52)
for name, arr in [('wofi_z',        wofi_z),
                   ('hawkes_z',      hawkes_z),
                   ('kyle_lambda_z', kyle_lambda_z),
                   ('amihud_z',      amihud_z),
                   ('spread_z',      spread_z)]:
    print(f'{name:<16} {arr.mean():>8.3f} {arr.std():>8.3f} '
          f'{arr.min():>8.2f} {arr.max():>8.2f}')
print()
print('✅ All features normalized to approximately [-3, 3] range')

In [ ]:
print('Creating forward-looking labels...')

HORIZONS = {
    'label_5s':   50,    # 5 seconds
    'label_30s':  300,   # 30 seconds ← KEY METRIC
    'label_5min': 3000,  # 5 minutes
}

mid_series = pl.Series('mid_price', mid_prices)
labels     = {}

for label_name, horizon in HORIZONS.items():
    future_mid = mid_series.shift(-horizon)
    label = (future_mid > mid_series).cast(pl.Int8)
    # Do NOT fill_null here — let the explicit drop handle it
    labels[label_name] = label.to_numpy()
    pct_up = np.nanmean(labels[label_name]) * 100  # nanmean ignores trailing NaNs
    print(f'  {label_name:<12} horizon={horizon:>4} ticks | {pct_up:.1f}% UP')

print()
print('⚠️  Labels ONLY belong in y arrays — never feed them as input features X.')



In [ ]:
# Cell 11: Assemble final feature DataFrame
# Exact output columns required by blueprint:
# [timestamp, symbol, mid_price, spread,
#  wofi, hawkes_intensity, kyle_lambda, amihud_illiq,
#  wofi_z, hawkes_z, kyle_lambda_z, amihud_z,
#  label_5s, label_30s, label_5min]
# Plus: realized_vol, autocorrelation, spread_z (for HMM + transformer)

print('Assembling feature DataFrame...')
t0 = time.time()

df_features = pl.DataFrame({
    # ── Identity ──────────────────────────────────────────────────
    'timestamp':        df['timestamp'],
    'symbol':           df['symbol'],
    # ── Raw features (interpretable, for analysis) ─────────────────
    'mid_price':        mid_prices.tolist(),
    'spread':           df['spread'],
    'wofi':             wofi_values.tolist(),
    'hawkes_intensity': hawkes_intensity.tolist(),
    'kyle_lambda':      kyle_lambda.tolist(),
    'amihud_illiq':     amihud_illiq.tolist(),
    # ── HMM context features (Notebook 04 inputs) ──────────────────
    'realized_vol':     realized_vol.tolist(),
    'autocorrelation':  autocorr_values.tolist(),
    # ── Normalized features (LOBTransformer model inputs) ───────────
    'wofi_z':           wofi_z.tolist(),
    'hawkes_z':         hawkes_z.tolist(),
    'kyle_lambda_z':    kyle_lambda_z.tolist(),
    'amihud_z':         amihud_z.tolist(),
    'spread_z':         spread_z.tolist(),
    # ── Labels (TARGET ONLY — never model inputs) ───────────────────
    'label_5s':         labels['label_5s'].tolist(),
    'label_30s':        labels['label_30s'].tolist(),
    'label_5min':       labels['label_5min'].tolist(),
})

# Drop the last 3001 rows — they have no valid 5min forward label
df_features = df_features.head(len(df_features) - 3001)

print(f'✅ Feature DataFrame assembled in {time.time()-t0:.1f}s')
print(f'   Shape:   {df_features.shape}')
print(f'   Columns: {df_features.columns}')

In [ ]:
# Cell 12: Save to parquet and verify

t0 = time.time()
df_features.write_parquet(PARQUET_OUT)
file_mb = os.path.getsize(PARQUET_OUT) / 1e6
elapsed = time.time() - t0

# Verify round-trip
df_verify = pl.read_parquet(PARQUET_OUT)
assert len(df_verify) == len(df_features),   'Row count mismatch on round-trip!'
assert df_verify.columns == df_features.columns, 'Column mismatch on round-trip!'
del df_verify

print(f'✅ Saved to {PARQUET_OUT}')
print(f'   File size: {file_mb:.0f} MB')
print(f'   Save time: {elapsed:.1f}s')
print(f'   Round-trip verified ✅')

In [ ]:
# Cell 13: Visualize feature distributions (raw vs normalized)

import matplotlib.pyplot as plt

sample = df_features.sample(10_000, seed=42)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('AlphaLOB — Feature Distributions: Raw vs Z-Score Normalized', fontsize=13, fontweight='bold')

raw_cols  = ['wofi',   'hawkes_intensity', 'kyle_lambda',    'amihud_illiq']
norm_cols = ['wofi_z', 'hawkes_z',         'kyle_lambda_z',  'amihud_z']
titles    = ['WOFI',   'Hawkes λ(t)',       "Kyle's Lambda",  'Amihud ILLIQ']
colors    = ['#2196F3','#4CAF50',           '#FF9800',        '#9C27B0']

for j, (raw, norm, title, color) in enumerate(zip(raw_cols, norm_cols, titles, colors)):
    # Raw distribution (top row)
    axes[0, j].hist(sample[raw].to_numpy(), bins=60, color=color, alpha=0.8)
    axes[0, j].set_title(f'{title} (raw)')
    axes[0, j].grid(alpha=0.3)

    # Z-score distribution (bottom row)
    axes[1, j].hist(sample[norm].to_numpy(), bins=60, color=color, alpha=0.8)
    axes[1, j].set_title(f'{title} (z-score)')
    axes[1, j].axvline(0,  color='red',  linestyle='--', alpha=0.6, label='μ=0')
    axes[1, j].axvline(-3, color='gray', linestyle=':',  alpha=0.5)
    axes[1, j].axvline(+3, color='gray', linestyle=':',  alpha=0.5, label='±3σ')
    axes[1, j].set_xlim(-5, 5)
    axes[1, j].legend(fontsize=7)
    axes[1, j].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/features_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature distributions saved → /content/features_overview.png')

In [ ]:
# Cell 14: Label balance check

print('=== LABEL BALANCE CHECK ===')
for lbl in ['label_5s', 'label_30s', 'label_5min']:
    up   = int(df_features[lbl].sum())
    down = len(df_features) - up
    pct  = up / len(df_features) * 100
    ok   = '✅' if 45 <= pct <= 55 else '⚠️'
    print(f'  {ok} {lbl:<12}: UP={up:,} ({pct:.1f}%) | DOWN={down:,} ({100-pct:.1f}%)')

print()
print('  Labels should be ~50/50. If badly skewed, add class_weight to the loss in Notebook 03.')

print()
print('=' * 58)
print('  NOTEBOOK 02 COMPLETE — FEATURE ENGINEERING')
print('=' * 58)
print(f'  Output: {PARQUET_OUT}')
print(f'  Rows:   {len(df_features):,}')
print(f'  Shape:  {df_features.shape}')
print()
print('  Features computed:')
print('    ✅ WOFI           — deque O(1) rolling window')
print('    ✅ Hawkes λ(t)    — MLE fit once, recursive apply')
print("    ✅ Kyle's Lambda  — rolling OLS 5-min (statsmodels)")
print('    ✅ Amihud ILLIQ   — rolling |ret|/dollar_vol')
print('    ✅ Z-Score norm   — rolling 1000-tick window, no look-ahead')
print('    ✅ Labels         — 3 horizons (5s, 30s, 5min)')
print()
print('  Next step → Run 03_train_lobtransformer.ipynb')
print('=' * 58)